# LangSmith Tracing — Observability for LangGraph Agents
## A Hands-On Workshop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/73-langsmith-tracing/langsmith_tracing_workbook.ipynb)

Instruments a LangGraph Q&A agent with LangSmith tracing via @traceable. Requires LANGSMITH_API_KEY and LANGSMITH_TRACING=true env vars. Shows how traces, spans, and metadata appear in the LangSmith UI.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why observability matters; LangSmith traces vs logs |
| 2 | **Setup** — LANGSMITH_API_KEY + LANGSMITH_TRACING=true; auto-instrumentation |
| 3 | **@traceable** — Decorator turns any function into a traced span |
| 4 | **LangGraph Auto-Tracing** — LangGraph pipelines traced automatically when env vars are set |
| 5 | **Full Pipeline** — Traced Q&A graph with named spans per node |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "langsmith", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

In [ ]:
import os
from typing import TypedDict
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langsmith import traceable
from langgraph.graph import END, START, StateGraph

# LangSmith activates automatically when these are set:
# os.environ["LANGSMITH_API_KEY"] = "lsv2_..."
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "my-project"

LANGSMITH_CONFIGURED = bool(os.environ.get("LANGSMITH_API_KEY"))

SAMPLE_QUESTIONS = [
    "What is retrieval-augmented generation?",
    "Explain the ReAct agent pattern in two sentences.",
    "What is LangSmith used for?",
]

class TraceState(TypedDict):
    question: str
    answer: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

@traceable(name="answer-node")
def answer(state: TraceState) -> dict:
    response = llm.invoke([HumanMessage(content=state["question"])])
    return {"answer": response.content.strip()}

g = StateGraph(TraceState)
g.add_node("answer", answer)
g.add_edge(START, "answer")
g.add_edge("answer", END)
app = g.compile()

print(f"LangSmith configured: {LANGSMITH_CONFIGURED}")
print(f"Tracing active: {os.environ.get('LANGSMITH_TRACING', 'false')}")
print()

for q in SAMPLE_QUESTIONS:
    result = app.invoke({"question": q, "answer": ""})
    print(f"Q: {q}")
    print(f"A: {result['answer'][:150]}")
    if LANGSMITH_CONFIGURED:
        print("  → Trace logged to LangSmith")
    print()

print("To enable tracing:")
print("  export LANGSMITH_API_KEY=lsv2_...")
print("  export LANGSMITH_TRACING=true")
print("  Then re-run — all graph calls appear in app.langsmith.com")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*